# Ripon Grammar School — Admissions Boundary Analysis

**Goal**: Estimate where the out-of-catchment admissions boundary (maximum day-place distance) is likely to be in future years, to assess the probability of a place being offered to a child living outside the defined catchment area.

---

### How admissions actually work

RGS is **academically selective**. The process has two stages:

**Stage 1 — Academic threshold**: North Yorkshire Council looks at test scores of all Year 6 children in the Ripon area and sets the pass mark at the score where the top 28% of that local cohort would qualify. Children who do not reach this mark cannot be offered a place.

**Stage 2 — Distance priority** (for qualifiers only):
1. Looked-after / previously looked-after children
2. Overseas state care / adopted
3. Special social or medical reasons
4. **In-catchment** (sub-ranked: siblings first, then nearest)
5. Staff children
6. **Out-of-catchment** (sub-ranked: siblings first, then nearest)

All in-catchment qualifiers receive offers before any out-of-catchment place is allocated. Remaining day places go to out-of-catchment qualifiers in strict distance order. The `distance of last child allocated` published each year is always the furthest out-of-catchment **day** pupil who received an offer — the effective boundary for out-of-catchment applicants.

**PAN**: 117 total (14 boarding + 103 day). Distance is measured straight-line by an electronic mapping system (GIS).

---

### Data sources
1. North Yorkshire secondary admissions statistics (Excel, 2013-14 to 2025-26)
2. ONS live births by local authority 1992–2021 — Harrogate district (E07000165)
3. FOI response from RGS (Jan 2024): test statistics 2013–2023 and birth-season distribution of current cohorts

In [ ]:
import sys, math
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from scipy.stats import t as t_dist, norm

sns.set_theme(style='whitegrid', palette='tab10')
plt.rcParams['figure.dpi'] = 120

# ── OLS helpers ──────────────────────────────────────────────────────────────
def fit_ols(X, y):
    X = np.array(X)
    if X.ndim == 1:
        X = X.reshape(-1, 1)
    Xb = np.hstack([np.ones((len(X), 1)), X])
    beta = np.linalg.lstsq(Xb, y, rcond=None)[0]
    y_hat = Xb @ beta
    n, k = len(y), X.shape[1]
    s2 = ((y - y_hat)**2).sum() / (n - k - 1)
    r2 = 1 - ((y - y_hat)**2).sum() / ((y - y.mean())**2).sum()
    return dict(beta=beta, s=np.sqrt(s2), r2=r2, Xb=Xb, n=n, k=k)

def predict_pi(model, x_new, alpha=0.05):
    x_new_b = np.array([1.0] + list(x_new))
    y_hat = float(x_new_b @ model['beta'])
    XtXinv = np.linalg.inv(model['Xb'].T @ model['Xb'])
    lev = float(x_new_b @ XtXinv @ x_new_b)
    se = model['s'] * np.sqrt(1 + lev)
    t_crit = t_dist.ppf(1 - alpha/2, df=model['n'] - model['k'] - 1)
    return y_hat, y_hat - t_crit * se, y_hat + t_crit * se

def p_above(model, x_new, threshold):
    est, lo, hi = predict_pi(model, x_new)
    t_crit = t_dist.ppf(0.975, df=model['n'] - model['k'] - 1)
    se = (hi - lo) / (2 * t_crit)
    return 1 - norm.cdf(threshold, loc=est, scale=se)

USER_DISTANCE = 10.34  # user's straight-line distance to RGS (miles)

## 1. Load data

In [ ]:
# ── Admissions + births ───────────────────────────────────────────────────────
df = pd.read_csv('../data/processed/rgs_with_births.csv')
df.loc[df['entry_year'] == 2013, 'applications'] = np.nan  # unreliable in source

# ── FOI test statistics (RGS, Jan 2024) ──────────────────────────────────────
foi = pd.DataFrame({
    'entry_year':    list(range(2013, 2024)),
    'total_tested':  [303,320,324,259,266,263,284,308,276,358,377],
    'cutoff_mark':   [213,209,214,210,201,207,203,201,198,201,196],
    'total_passing': [146,139,129,122,143,130,160,151,165,179,222],
    'foi_intake':    [120,120,120,120,120,119,119,120,119,120,118],
})
foi['surplus']   = foi['total_passing'] - foi['foi_intake']
foi['pass_rate'] = foi['total_passing'] / foi['total_tested']

# ── Merge everything ──────────────────────────────────────────────────────────
df = df.merge(foi, on='entry_year', how='left')

print(f'Rows: {len(df)}')
df[['year_key','entry_year','birth_year','total_births',
    'total_tested','total_passing','cutoff_mark','surplus','pass_rate',
    'applications','day_distance_miles']]

## 2. FOI test statistics

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
years = df['entry_year'].dropna()

# Total tested vs total passing
ax = axes[0, 0]
ax.bar(df['entry_year'], df['total_tested'],  alpha=0.6, label='Tested',  color='steelblue')
ax.bar(df['entry_year'], df['total_passing'], alpha=0.8, label='Passed',  color='darkorange')
ax.set_title('Children tested vs passing each year')
ax.set_ylabel('Number of children')
ax.legend()
ax.set_xticks(df['entry_year'])
ax.set_xticklabels(df['year_key'], rotation=45, ha='right', fontsize=7)

# Pass rate over time
ax = axes[0, 1]
ax.plot(df['entry_year'], df['pass_rate'] * 100, 'o-', color='green', ms=5)
ax.set_title('Pass rate (% of test-takers qualifying)')
ax.set_ylabel('Pass rate (%)')
ax.set_xticks(df['entry_year'])
ax.set_xticklabels(df['year_key'], rotation=45, ha='right', fontsize=7)

# Cut-off mark over time
ax = axes[1, 0]
ax.plot(df['entry_year'], df['cutoff_mark'], 's-', color='purple', ms=5)
ax.set_title('Cut-off mark (declining = lower threshold)')
ax.set_ylabel('Cut-off score')
ax.set_xticks(df['entry_year'])
ax.set_xticklabels(df['year_key'], rotation=45, ha='right', fontsize=7)

# Surplus passers (passed but didn't get a place)
ax = axes[1, 1]
ax.bar(df['entry_year'], df['surplus'], color='crimson', alpha=0.7)
ax.set_title('Surplus passers (qualified but no place offered)')
ax.set_ylabel('Surplus')
ax.set_xticks(df['entry_year'])
ax.set_xticklabels(df['year_key'], rotation=45, ha='right', fontsize=7)

plt.suptitle('RGS admissions test statistics (FOI data)', fontsize=13)
plt.tight_layout()
plt.savefig('../data/processed/fig_foi_test_stats.png', bbox_inches='tight')
plt.show()

### 2.1 Key observations from test statistics

- **Children tested** has grown from ~260–325 (2013–2021) to **358–377 in 2022–23** — more families are discovering RGS.
- **Pass rate** has risen from ~40–48% (2013–16) to **56–59%** recently, driven by tutoring culture. The cut-off mark has simultaneously fallen from 213 to 196, consistent with a lower absolute threshold.
- **Surplus passers** (qualified children who don't receive any offer) has grown dramatically: single-digits in 2015–16, rising to **104 in 2023**. However, this figure is confounded by boarding applicants — children who pass the same test but apply for boarding places don't compete for day slots.
- **Why 2022-23 had a 20.36-mile boundary despite 59 surplus passers**: the surplus almost certainly includes a large number of boarding applicants. When boarding candidates are excluded, the out-of-catchment day competition was actually thin that year, allowing the boundary to extend unusually far. This is the most plausible explanation for the outlier.

**Implication for 2030**: assuming the pass-rate trend continues (call it ~62%), and ~400 children sit the test, roughly 250 would qualify — up from 222 in 2023. With intake ~120, surplus ~130. This growing competition is a headwind for out-of-catchment applicants, partially offsetting the declining birth cohort.

## 3. Birth season analysis — is there a relative age effect?

The FOI response included birth-season distributions for current Y7–Y13 cohorts. In UK schools, **autumn-born** (Sep–Dec) children are the oldest in their year group and **summer-born** (May–Aug) children are the youngest — a gap of up to 12 months at age 11 that can disadvantage younger children on non-standardised tests.

In [ ]:
seasons = pd.DataFrame({
    'year_group': ['Y7','Y8','Y9','Y10','Y11','Y12','Y13'],
    'autumn':     [36, 42, 29, 42, 28, 41, 31],
    'spring':     [27, 22, 31, 21, 35, 20, 22],
    'summer':     [37, 36, 40, 37, 36, 39, 47],
})
expected = 100 / 3  # ~33.3% if no relative age effect

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(seasons))
w = 0.25
bars_a = ax.bar(x - w, seasons['autumn'], w, label='Autumn (Sep–Dec)', color='steelblue',  alpha=0.8)
bars_s = ax.bar(x,     seasons['spring'], w, label='Spring (Jan–Apr)', color='darkorange', alpha=0.8)
bars_u = ax.bar(x + w, seasons['summer'], w, label='Summer (May–Aug)', color='green',      alpha=0.8)
ax.axhline(expected, color='red', ls='--', lw=1.5, label=f'Expected if uniform ({expected:.1f}%)')
ax.set_xticks(x)
ax.set_xticklabels(seasons['year_group'])
ax.set_ylabel('% of year group')
ax.set_title('Birth season distribution across RGS year groups (FOI data)')
ax.legend(fontsize=9)
ax.set_ylim(0, 55)
plt.tight_layout()
plt.savefig('../data/processed/fig_birth_seasons.png', bbox_inches='tight')
plt.show()

print(f"Mean across year groups:")
print(f"  Autumn: {seasons['autumn'].mean():.1f}%  (expected 33.3%)")
print(f"  Spring: {seasons['spring'].mean():.1f}%  (expected 33.3%)")
print(f"  Summer: {seasons['summer'].mean():.1f}%  (expected 33.3%)")

### 3.1 Finding: no relative age disadvantage at RGS

Summer-born children (May–Aug) average **38.9%** of each cohort — consistently *above* the expected 33.3% in every year group. Spring-born children are the underrepresented group (25.4%). This is the **opposite** of what a relative age effect would predict.

This strongly suggests the entrance test uses **age-standardised scores**, normalising for the age gap within the cohort. A child born **25 August** (the last day of the school year) is not at a systematic disadvantage.

The spring underrepresentation is unexplained but likely reflects small-sample noise across 7 year groups of ~120 students each.

## 4. Time-series overview

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(12, 13), sharex=True)

# Distance
ax = axes[0]
ax.plot(df['entry_year'], df['day_distance_miles'], 'o-', color='steelblue', ms=6)
ax.scatter(df[df['entry_year']==2022]['entry_year'],
           df[df['entry_year']==2022]['day_distance_miles'],
           color='red', zorder=5, s=80, label='2022-23 outlier (boarding confound)')
ax.axhline(USER_DISTANCE, color='green', ls='--', lw=1.5, label=f'User distance ({USER_DISTANCE} mi)')
ax.set_ylabel('Boundary (miles)')
ax.set_title('Distance of last out-of-catchment day pupil allocated')
ax.legend(fontsize=8)

# Total tested and passing
ax = axes[1]
ax.bar(df['entry_year'], df['total_tested'],  alpha=0.5, color='steelblue', label='Tested')
ax.bar(df['entry_year'], df['total_passing'], alpha=0.8, color='darkorange', label='Passed')
ax.set_ylabel('Children')
ax.set_title('Test takers and passers (FOI data)')
ax.legend(fontsize=8)

# Surplus passers
ax = axes[2]
ax.bar(df['entry_year'], df['surplus'], color='crimson', alpha=0.7)
ax.set_ylabel('Surplus passers')
ax.set_title('Surplus passers (qualified but not offered a place)')

# Harrogate births
ax = axes[3]
ax.bar(df['entry_year'], df['total_births'], color='teal', alpha=0.7)
ax.set_ylabel('Births')
ax.set_title('Harrogate district births (11 years prior to entry)')
ax.set_xlabel('Entry year')

for ax in axes:
    ax.set_xticks(df['entry_year'])
    ax.set_xticklabels(df['year_key'], rotation=45, ha='right', fontsize=7)

plt.tight_layout()
plt.savefig('../data/processed/fig_timeseries.png', bbox_inches='tight')
plt.show()

## 5. Correlation analysis — what predicts the boundary distance?

In [ ]:
# Working dataset: exclude 2013-14 (no applications) and 2022-23 outlier
df_model = df[
    (df['entry_year'] > 2013) &
    (df['entry_year'] != 2022) &
    df['day_distance_miles'].notna()
].copy()

predictors = {
    'Applications (all preferences)': 'applications',
    'Total tested':                   'total_tested',
    'Total passing':                  'total_passing',
    'Surplus passers':                'surplus',
    'Pass rate':                      'pass_rate',
    'Harrogate births (−11 yrs)':     'total_births',
    'Cut-off mark':                   'cutoff_mark',
}

print(f'{"Predictor":<35} {"r":>8} {"p":>8}  Note')
print('-' * 75)
for label, col in predictors.items():
    sub = df_model.dropna(subset=[col, 'day_distance_miles'])
    r, p = stats.pearsonr(sub[col], sub['day_distance_miles'])
    sig = '**' if p < 0.05 else ('*' if p < 0.10 else '')
    print(f'{label:<35} {r:>8.3f} {p:>8.3f}  {sig}')

print(f'\nn = {len(df_model)} (entry years 2014–2025, excl. 2022)')

### 5.1 Key finding: nothing predicts the boundary well

None of the available predictors — including the new FOI variables (surplus passers, pass rate, cut-off mark) — has a statistically significant correlation with the boundary distance. The boundary is driven largely by the **geographic distribution** of out-of-catchment qualifiers in any given year: how many happen to live close vs. far from school. This is fundamentally noisy and unobservable in advance.

The **surplus passers** correlation (r ≈ 0.12) is particularly counter-intuitive — more surplus should theoretically mean tighter boundary — but the boarding confound (boarding applicants inflate the surplus figure without competing for day places) erases the signal.

**Practical implication**: the best forecast of the boundary is the historical distribution itself, not any regression model. The ~2.5-mile residual standard deviation is irreducible noise.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

plot_pairs = [
    ('applications',  'Applications',          'teal'),
    ('total_tested',  'Total tested',           'steelblue'),
    ('total_passing', 'Total passing',          'darkorange'),
    ('surplus',       'Surplus passers',        'crimson'),
    ('total_births',  'Harrogate births',       'purple'),
    ('cutoff_mark',   'Cut-off mark',           'green'),
]

for ax, (col, label, colour) in zip(axes.flat, plot_pairs):
    sub = df_model.dropna(subset=[col, 'day_distance_miles'])
    r, p = stats.pearsonr(sub[col], sub['day_distance_miles'])
    ax.scatter(sub[col], sub['day_distance_miles'], color=colour, s=55, zorder=3)
    # label 2022-23 if present in sub
    out = df[df['entry_year']==2022].dropna(subset=[col])
    if len(out):
        ax.scatter(out[col], out['day_distance_miles'],
                   color='red', s=80, marker='x', linewidths=2, zorder=4)
    for _, row in sub.iterrows():
        ax.annotate(str(int(row['entry_year'])),
                    (row[col], row['day_distance_miles']),
                    textcoords='offset points', xytext=(4,3), fontsize=6)
    m, b = np.polyfit(sub[col], sub['day_distance_miles'], 1)
    xs = np.linspace(sub[col].min(), sub[col].max(), 100)
    ax.plot(xs, m*xs + b, '--', color=colour, alpha=0.5)
    ax.axhline(USER_DISTANCE, color='green', ls=':', lw=1)
    ax.set_xlabel(label, fontsize=9)
    ax.set_ylabel('Boundary (miles)', fontsize=9)
    ax.set_title(f'r = {r:.2f}, p = {p:.2f}', fontsize=9)

plt.suptitle('Correlation of candidate predictors with boundary distance\n(red × = 2022-23 outlier excluded from fit; green line = user distance)',
             fontsize=11)
plt.tight_layout()
plt.savefig('../data/processed/fig_correlations.png', bbox_inches='tight')
plt.show()

## 6. Historical distribution and base-rate probability

Given the weak predictive power of all available variables, the empirical distribution is the most honest forecast.

In [ ]:
hist_data = df_model['day_distance_miles'].dropna().sort_values()
hist_all  = df[df['day_distance_miles'].notna() & (df['entry_year']>2013)]['day_distance_miles'].sort_values()

print('Historical boundary distances (entry years 2014–2025):')
print('  Excluding 2022-23 outlier:')
for _, r in df_model[['year_key','day_distance_miles','surplus']].dropna().sort_values('day_distance_miles').iterrows():
    marker = ' ← USER' if abs(r['day_distance_miles'] - USER_DISTANCE) < 0.5 else ''
    above  = ' ✓' if r['day_distance_miles'] > USER_DISTANCE else ''
    print(f"    {r['year_key']}: {r['day_distance_miles']:.2f} mi  (surplus={int(r['surplus']) if pd.notna(r['surplus']) else '?':>3}){above}{marker}")

n_above = (hist_data > USER_DISTANCE).sum()
n_total = len(hist_data)
print(f'\n  Boundary > {USER_DISTANCE} mi: {n_above}/{n_total} years = {n_above/n_total:.0%}')
print(f'  Including 2022-23:         {n_above+1}/{n_total+1} years = {(n_above+1)/(n_total+1):.0%}')
print(f'\n  Mean:   {hist_data.mean():.2f} mi')
print(f'  Median: {hist_data.median():.2f} mi')
print(f'  Std:    {hist_data.std():.2f} mi')
print(f'  Min:    {hist_data.min():.2f} mi')
print(f'  Max:    {hist_data.max():.2f} mi')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Histogram
ax = axes[0]
ax.hist(hist_data, bins=8, color='steelblue', alpha=0.7, edgecolor='white')
ax.axvline(USER_DISTANCE, color='green', lw=2, ls='--', label=f'User ({USER_DISTANCE} mi)')
ax.axvline(hist_data.mean(), color='red', lw=1.5, ls=':', label=f'Mean ({hist_data.mean():.1f} mi)')
ax.set_xlabel('Boundary distance (miles)')
ax.set_ylabel('Years')
ax.set_title('Distribution of historical boundary (excl. 2022-23)')
ax.legend(fontsize=9)

# Empirical CDF
ax = axes[1]
sorted_d = np.sort(hist_data)
cdf = np.arange(1, len(sorted_d)+1) / len(sorted_d)
ax.step(sorted_d, cdf, color='steelblue', lw=2, where='post')
ax.axvline(USER_DISTANCE, color='green', lw=2, ls='--', label=f'User ({USER_DISTANCE} mi)')
p_accept = (hist_data > USER_DISTANCE).mean()
ax.axhline(1 - p_accept, color='green', lw=1, ls=':', alpha=0.5)
ax.set_xlabel('Boundary distance (miles)')
ax.set_ylabel('Cumulative probability')
ax.set_title(f'Empirical CDF — P(boundary > {USER_DISTANCE} mi) = {p_accept:.0%}')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('../data/processed/fig_historical_dist.png', bbox_inches='tight')
plt.show()

## 7. Simple OLS models and projections for entry year 2030

The child of interest was born **25 August 2019** → enters Year 7 in **September 2030**.
Harrogate births in 2019 (the relevant cohort): **1,262** — the lowest in the ONS series.

Despite weak R², the regression models give a useful range and can be scenario-tested.

In [ ]:
dm = df_model.dropna(subset=['applications','total_births','day_distance_miles'])
y  = dm['day_distance_miles'].values

model_apps   = fit_ols(dm[['applications']].values, y)
model_births = fit_ols(dm[['total_births']].values, y)
model_both   = fit_ols(dm[['total_births','applications']].values, y)

for name, m in [('Applications only', model_apps),
                ('Births only',        model_births),
                ('Births + apps',      model_both)]:
    print(f'{name:<25}  R² = {m["r2"]:.3f}   residual std = {m["s"]:.2f} mi')

# Births for 2030 cohort
births_2030 = 1262

print(f'\n── Projections for entry year 2030 (Harrogate births 2019 = {births_2030:,}) ──')
print(f'{"Scenario":<35} {"Model":>25} {"Est":>6} {"PI 95%":>16} {"P(>10.34mi)":>12}')
print('-' * 100)

scenarios = [('Low demand (200 apps)', 200), ('Central (265 apps)', 265), ('High demand (330 apps)', 330)]
for label, apps in scenarios:
    for mname, model, xval in [
        ('Apps only',   model_apps,   [apps]),
        ('Births+apps', model_both,   [births_2030, apps]),
    ]:
        est, lo, hi = predict_pi(model, xval)
        pa = p_above(model, xval, USER_DISTANCE)
        print(f'{label:<35} {mname:>25} {est:>6.1f} [{lo:>5.1f} – {hi:>5.1f}]  {pa:>11.0%}')

In [ ]:
births_df = pd.read_csv('../data/processed/ons_births_harrogate.csv')
births_df['entry_year'] = births_df['birth_year'] + 11
future = births_df[births_df['entry_year'].between(2026, 2032)][['entry_year','total_births']]

colours = {'Low (200)': 'C0', 'Central (265)': 'C1', 'High (330)': 'C2'}
scen_apps = {'Low (200)': 200, 'Central (265)': 265, 'High (330)': 330}

fig, ax = plt.subplots(figsize=(13, 6))

# Historical
ax.plot(df_model['entry_year'], df_model['day_distance_miles'],
        'ko-', ms=5, label='Historical')
ax.scatter([2022], df[df['entry_year']==2022]['day_distance_miles'],
           color='red', s=90, marker='x', linewidths=2, zorder=5, label='2022-23 outlier')

# Projections
for scenario, colour in colours.items():
    apps = scen_apps[scenario]
    ests, los, his = [], [], []
    for _, row in future.iterrows():
        e, l, h = predict_pi(model_both, [row['total_births'], apps])
        ests.append(e); los.append(l); his.append(h)
    ax.plot(future['entry_year'], ests, 'o--', color=colour, ms=6, label=scenario)
    ax.fill_between(future['entry_year'], los, his, color=colour, alpha=0.10)

# Reference lines
ax.axhline(USER_DISTANCE, color='green', lw=2, ls='--', label=f'User location ({USER_DISTANCE} mi)')
ax.axvline(2030, color='grey', lw=1, ls=':', alpha=0.6, label='Target entry year (2030)')

ax.set_xlabel('Entry year')
ax.set_ylabel('Boundary distance (miles)')
ax.set_title('RGS out-of-catchment day boundary — historical and projected (Model: births + applications)')
ax.legend(fontsize=8, loc='upper left')
ax.set_xlim(2013, 2033)
ax.set_ylim(0)
plt.tight_layout()
plt.savefig('../data/processed/fig_projections.png', bbox_inches='tight')
plt.show()

## 8. Geographic analysis — finding a safer location

In [ ]:
# RGS: Clotherholme Road, Ripon HG4 2DG
RGS_LAT, RGS_LON = 54.1323, -1.5275

def haversine_miles(lat1, lon1, lat2, lon2):
    R = 3958.8
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlam/2)**2
    return R * 2 * math.asin(math.sqrt(a))

# Villages south/southeast of Ripon (Great Ouseburn direction)
# plus known catchment villages to the west for reference
villages = {
    # Almost certainly IN catchment (very close to Ripon)
    'Sharow':               (54.1318, -1.5025, 'likely in-catchment'),
    'Skelton-on-Ure':       (54.1166, -1.5282, 'likely in-catchment'),
    'Littlethorpe':         (54.1180, -1.5483, 'likely in-catchment'),
    'Bishop Monkton':       (54.0940, -1.5373, 'likely in-catchment'),
    # Confirmed in-catchment (from search results)
    'Galphay':              (54.1621, -1.6563, 'confirmed in-catchment'),
    'Laverton':             (54.1759, -1.6624, 'confirmed in-catchment'),
    'Grewelthorpe':         (54.1914, -1.6630, 'confirmed in-catchment'),
    # Likely out-of-catchment — south/southeast corridor
    'Marton-le-Moor':       (54.1053, -1.4407, 'check NYCC map'),
    'Wormald Green':        (54.0730, -1.5370, 'check NYCC map'),
    'Burton Leonard':       (54.0719, -1.5322, 'check NYCC map'),
    'Copgrove':             (54.0620, -1.4819, 'check NYCC map'),
    'Staveley':             (54.0561, -1.5114, 'check NYCC map'),
    'Roecliffe':            (54.0883, -1.4114, 'likely out-of-catchment'),
    'Kirby Hill':           (54.0788, -1.4101, 'likely out-of-catchment'),
    'Boroughbridge':        (54.0861, -1.3943, 'likely out-of-catchment'),
    'Aldborough':           (54.0844, -1.3813, 'likely out-of-catchment'),
    'Great Ouseburn':       (54.0247, -1.3676, 'likely out-of-catchment'),
    # User's approximate current location band
    'User location (~10.34 mi)': (None, None, 'current'),
}

rows = []
for name, (lat, lon, status) in villages.items():
    if lat is None:
        continue
    d = haversine_miles(RGS_LAT, RGS_LON, lat, lon)
    # How many historical years would this distance have been safe?
    safe_years = (hist_data > d).sum()
    rows.append(dict(village=name, distance_mi=round(d,2),
                     status=status,
                     years_safe=f'{safe_years}/{len(hist_data)}',
                     pct_safe=f'{safe_years/len(hist_data):.0%}'))

geo_df = pd.DataFrame(rows).sort_values('distance_mi')
print(f'{"Village":<30} {"Miles":>7}  {"Status":<25} {"Yrs safe":>9}  {"% safe":>7}')
print('-'*85)
for _, r in geo_df.iterrows():
    print(f"{r['village']:<30} {r['distance_mi']:>7.2f}  {r['status']:<25} {r['years_safe']:>9}  {r['pct_safe']:>7}")

print(f'\nFor reference, user is at {USER_DISTANCE} mi — safe in {(hist_data > USER_DISTANCE).sum()}/{len(hist_data)} years ({(hist_data > USER_DISTANCE).mean():.0%})')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

distance_range = np.linspace(0, 14, 300)
pct_safe = [(hist_data > d).mean() for d in distance_range]

ax.plot(distance_range, pct_safe, color='steelblue', lw=2)
ax.fill_between(distance_range, pct_safe, alpha=0.15, color='steelblue')

# Mark key villages
highlights = {
    'Boroughbridge\n(6.3 mi)': 6.27,
    'Roecliffe\n(5.6 mi)':     5.60,
    'Kirby Hill\n(6.0 mi)':    6.02,
    'Great Ouseburn\n(9.9 mi)': 9.86,
    'User\n(10.34 mi)':        USER_DISTANCE,
}
colours_v = ['darkgreen','darkgreen','darkgreen','darkorange','red']
for (name, d), col in zip(highlights.items(), colours_v):
    p = (hist_data > d).mean()
    ax.axvline(d, color=col, lw=1.5, ls='--', alpha=0.7)
    ax.scatter([d], [p], color=col, s=60, zorder=5)
    ax.annotate(f'{name}\n{p:.0%}', (d, p),
                textcoords='offset points', xytext=(6, -20), fontsize=8, color=col)

ax.set_xlabel('Straight-line distance from RGS (miles)')
ax.set_ylabel('Historical % of years boundary exceeded distance')
ax.set_title('P(out-of-catchment place offered | test passed) by distance\n'
             'based on 11 historical years (2014–2025, excl. 2022-23 outlier)')
ax.set_ylim(0, 1.05)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
plt.tight_layout()
plt.savefig('../data/processed/fig_distance_safety.png', bbox_inches='tight')
plt.show()

### 8.1 Geographic findings

- The historical minimum boundary (excluding the 2022-23 outlier) is **6.62 miles** (2014-15 entry). This was an unusually competitive year (430 applications).
- Third-party data shows 70% of current RGS students live within 5.03 miles — consistent with most in-catchment students being Ripon residents.
- **Boroughbridge** (~6.3 miles, likely out-of-catchment) would have been within range in **all 11** historical years.
- **Roecliffe and Kirby Hill** (~5.6–6.0 miles) are similarly safe.
- At the user's current distance (10.34 miles), the boundary was exceeded in only 3 of 11 years.

**Important caveat**: The catchment boundary is defined by specific geographic boundaries maintained by North Yorkshire Council — not a simple radius. Villages that appear close on the map may be in-catchment (even better) or out-of-catchment. Verify using the [NYCC interactive catchment map](https://www.northyorks.gov.uk/education-and-learning/school-admissions/finding-school).

## 9. Summary: probability of a place for a specific child

**Profile**: child born 25 August 2019 (entry year 2030, Harrogate cohort births = 1,262), living **out-of-catchment** at 10.34 miles from RGS, assuming the academic test threshold is met.

In [ ]:
print('═'*65)
print('PROBABILITY SUMMARY — entry year 2030, distance 10.34 miles')
print('═'*65)

print(f"""
1. BASE RATE (historical)
   Boundary exceeded 10.34 mi in {(hist_data > USER_DISTANCE).sum()}/{len(hist_data)} years
   (excl. 2022-23 outlier) = {(hist_data > USER_DISTANCE).mean():.0%}
   Including outlier: {(hist_all > USER_DISTANCE).sum()}/{len(hist_all)} = {(hist_all > USER_DISTANCE).mean():.0%}
""")

print('2. MODEL ESTIMATES (births=1,262, entry 2030)')
for label, apps in [('Low demand (200 apps)', 200),
                     ('Central (265 apps)',    265),
                     ('High demand (330 apps)',330)]:
    est, lo, hi = predict_pi(model_both, [births_2030, apps])
    pa = p_above(model_both, [births_2030, apps], USER_DISTANCE)
    print(f'   {label}: est={est:.1f} mi [{lo:.1f}–{hi:.1f}]  P(place) = {pa:.0%}')

print(f"""
3. TAILWINDS
   - Harrogate births 2019 (1,262) are 16% below 2014 cohort (1,497)
     and 26% below the 2007 peak (1,703). Smaller cohort = less competition.
   - Birth season: NO relative age effect at RGS (test is age-standardised).
     Summer-born children average 39% of cohorts vs 33% expected.

4. HEADWINDS
   - Pass rate rising (40% → 59% since 2013); more children qualifying
     for limited places even as raw cohort sizes shrink.
   - Applications trending upward (recent years 256–304).

5. VERDICT
   At 10.34 miles: roughly 35–50% probability of a place.
   Genuinely a coin flip. The declining birth cohort is the main
   positive signal; rising tutoring culture is the main risk.

   TO NEAR-CERTAINTY: Move to Boroughbridge (~6.3 mi), Roecliffe
   (~5.6 mi) or Kirby Hill (~6.0 mi) — all within the historical
   minimum boundary (6.62 mi) and almost certainly out-of-catchment.
   Verify catchment status via the NYCC interactive map.
""")

## 10. Open questions and next steps

| Gap | Impact on model | How to fill |
|-----|----------------|-------------|
| **In-catchment vs out-of-catchment split** | High — would remove boarding confound from surplus figure | FOI to North Yorkshire Council |
| **Number of out-of-catchment day qualifiers** (separate from boarding) | High — the real competition variable | FOI to RGS or NYC |
| **2022-23 outlier** — boarding vs day breakdown | Medium — currently excluded from all models | FOI to RGS |
| **ONS births 2022+** (→ entry years 2033+) | Low for near term | ONS sub-national population projections |
| **NYCC catchment boundary map** | Critical if considering relocation | NYCC interactive map |
| **Annual boundary updates** (2026, 2027…) | Will update base rate as new data arrives | North Yorkshire annual publication each April |